# Valutazione degli Encoder e Data Leak in Unsupervised Domain Adaptation (UDA)

In questo notebook esploriamo le diverse architetture disponibili per il nostro estrattore di feature (l'encoder). L'obiettivo della Track 9 è l'Unsupervised Domain Adaptation: adattare un modello allenato su domini *sorgente* (HMDB51, UCF101) a un dominio *target* (Kinetics-400) senza usare le etichette di Kinetics.

## Il Problema del "Data Leak" (O come non barare)
La maggior parte delle CNN 3D (come `r2plus1d_18`) disponibili in `torchvision` sono pre-addestrate proprio su **Kinetics-400**. Usare questi pesi pre-addestrati inizializza la rete con feature che hanno già "visto" e imparato le classi di Kinetics in modo supervisionato. In un contesto UDA rigoroso, questo è considerato un **Data Leak** (una fuga di dati dal target). 

### Le Metodologie per "Non Barare":
1. **Addestramento da zero (From Scratch):** Impostare `pretrained=False`. Purtroppo, le ResNet 3D hanno milioni di parametri. Allenarle da zero su dataset piccoli come HMDB/UCF porta a un *overfitting* catastrofico (l'accuratezza sul target crolla a ~1.5%). Per questo abbiamo introdotto la `Handcrafted3DCNN`, che avendo pochissimi parametri può essere addestrata da zero.
2. **Uso di Reti Leggerissime:** Architetture come `S3D` (~8M parametri) o `X3D_S` (~3M parametri) offrono un compromesso migliore. Possono essere addestrate da zero con maggiore successo rispetto alla ResNet (~33M parametri).
3. **Pesi Pre-addestrati su Altri Dataset:** L'opzione "eticamente pura" per usare pesi pre-addestrati è scaricare pesi allenati su enormi dataset *diversi* dal target, come **Sports-1M**, **Something-Something V2**, o **IG65M**. PyTorch non li fornisce nativamente, ma sono reperibili in rete.
4. **Congelamento Estremo (Compromesso Accademico):** Se si usano i pesi di Kinetics per necessità tecniche (mancanza di risorse computazionali), la prassi accademica tollerata è **congelare quasi tutta la rete** (Transfer Learning Rule 3) e sbloccare solo i layer finali per la fase di adattamento, dichiarandolo esplicitamente nel report.

---

## Rassegna delle Opzioni Architetturali

Di seguito un riepilogo delle opzioni supportate dal nostro `VideoEncoder` e le relative strategie di *fine-tuning*:

| Architettura | Parametri | Pre-trained | Pro/Contro | Fine-tuning |
|--------------|-----------|-------------|-------------------------|-------------------------|
| **Handcrafted3DCNN** | Pochi | No - from scratch | Addestramento rapido e da zero 3D CNN senza overfitting o data leak. Prestazioni assolute basse. | **Completo** (Full) |
| **r2plus1d_34_ig65m** | ~33M | IG-65M (video Instagram). Zero Kinetics. | Feature video robuste senza invalidare l'UDA. Troppi parametri per training da zero. | **Parziale** (Solo layer4) |
| **mvit_2d_imagenet / swin_2d_imagenet** | ~31M | ImageNet-1K (solo foto). Zero Kinetics. | Transformer (Multiscale Vision e Swin) nella loro variante 2D. Analizzano i frame uno ad uno e poi ne fanno la media. Molto potenti spazialmente, ottimi se il task dipende più dall'aspetto visivo che dal movimento rapido. | **Parziale** (Solo layer4) |
| **MC3_18** | ~11M | Kinetics | Usa conv 3D solo all'inizio e 2D alla fine. Molto più leggera delle ResNet standard. | **Completo** |
| **S3D / X3D_S** | ~3-8M | Kinetics | Leggere. Separa le conv spaziali e temporali. | **Completo** (da addestrare da zero) |

In [1]:
import sys
import os
import time

import torch
import torch.nn as nn

# Aggiungiamo la cartella src al path per importare VideoEncoder dal notebook
sys.path.append(os.path.abspath('../../'))
try:
    from src.models.backbone import VideoEncoder
    print("🚀 Modulo 'VideoEncoder' importato con successo dal framework di progetto!")
except ImportError as e:
    print(f"❌ Errore di importazione: {e}")
    print("Verifica la struttura delle cartelle.")

🚀 Modulo 'VideoEncoder' importato con successo dal framework di progetto!


In [2]:
# Funzione helper per contare e stampare i parametri
# ANALISI STRUTTURALE DELL'ENCODER
def analyze_encoder(model_type, pretrained=True, target_dim=512):
    print(f"\n{'='*70}")
    print(f"Architettura: {model_type.upper()} | Pretrained: {pretrained}")
    
    try:
        # Inizializzazione dell'encoder
        encoder = VideoEncoder(pretrained=pretrained, model_type=model_type)
    except Exception as e:
        print(f"Errore caricamento del modello {model_type}: {e}")
        return
        
    # Applichiamo la logica di congelamento 
    # Demandiamo il freezing parziale direttamente all'encoder
    # se il metodo è implementato, altrimenti usiamo una logica robusta basata sulle proiezioni finali.
    if hasattr(encoder, 'freeze_for_finetuning'):
        encoder.freeze_for_finetuning()
    else:
        # Fallback di simulazione se la logica è esterna
        if pretrained and model_type != "handcrafted":
            for param in encoder.parameters():
                param.requires_grad = False
                
            # Sblocco lo strato di pooling/proiezione lineare comune che uniforma a 512
            if hasattr(encoder, 'proj'):
                for param in encoder.proj.parameters():
                    param.requires_grad = True
            elif hasattr(encoder, 'fc'):
                for param in encoder.fc.parameters():
                    param.requires_grad = True
                    
            # Sblocco l'ultimo blocco convoluzionale profondo per le ResNet (layer4)
            # Questo permette l'estrazione di feature specifiche per il dominio sorgente
            if hasattr(encoder, 'backbone') and hasattr(encoder.backbone, 'layer4'):
                for param in encoder.backbone.layer4.parameters():
                    param.requires_grad = True

    # Calcolo della distribuzione dei parametri
    total_params = sum(p.numel() for p in encoder.parameters())
    trainable_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    print(f"Parametri totali:       {total_params:,}")
    print(f"Parametri addestrabili: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")
    print(f"Parametri congelati:    {frozen_params:,}")
    
    # Verification Forward Pass: Formato video standard Persona 1 (Batch=2, clip da 16 frame, 3 canali, 112x112)
    # Shape attesa del tensore in ingresso: (B, T, C, H, W)
    dummy_input = torch.randn(8, 16, 3, 112, 112)
    start_time = time.time()

    encoder.eval()
    with torch.no_grad():
        try:
            output = encoder(dummy_input)
            end_time = time.time()
            print(f"\n ✅ Test Forward Pass completato con successo!")
            print(f"    📊 Shape dell'Output: {output.shape})")
            print(f"    ⏱️ Tempo di inferenza (CPU): {(end_time - start_time):.3f} secondi")
            
            # Controllo dimensionale di sicurezza per evitare anomalie nel modulo delle Loss
            assert output.shape == (8, target_dim), f"Anomalia nella shape finale! Trovata: {output.shape}"
        except Exception as forward_error:
            print(f"❌ Errore durante il Forward Pass: {forward_error}")


In [3]:
# Testiamo Handcrafted 3d cnn (Training da zero, nessun parametro congelato)
# (Nessun data leak, addestramento completo 'from scratch')
analyze_encoder("handcrafted", pretrained=False) # config A / Baseline

# Testiamo architetture classiche (Fine-tuning parziale del layer 4)
analyze_encoder("r2plus1d_34_ig65m", pretrained=True) # Config B
analyze_encoder("mc3_18", pretrained=False)
# Testiamo l'architettura leggera S3D (Fine-tuning completo)
# (trade-off per addestramento 'from scratch' senza leak)
analyze_encoder("s3d", pretrained=False)
# Testiamo X3D
analyze_encoder("x3d_s", pretrained=False) # Config C

# Testiamo i Video Transformers 
#  MViT con pesi pre-addestrati su immagini 2d e ottimizzazione Dropout + Resizing inserita
analyze_encoder("mvit_2d_imagenet", pretrained=True)
# Swin3D
analyze_encoder("swin_2d_imagenet", pretrained=True)


Architettura: HANDCRAFTED | Pretrained: False
Inizializzazione della rete Handcrafted3DCNN from scratch (no pretraining)...
Parametri totali:       4,968,384
Parametri addestrabili: 4,968,384 (100.00%)
Parametri congelati:    0

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 1.088 secondi

Architettura: R2PLUS1D_34_IG65M | Pretrained: True
Caricamento R(2+1)D_34 preaddestrato su IG-65M (Puro, No Kinetics) via PyTorch Hub...


Using cache found in C:\Users\macca/.cache\torch\hub\moabitcoin_ig65m-pytorch_master
Downloading: "https://github.com/moabitcoin/ig65m-pytorch/releases/download/v1.0.0/r2plus1d_34_clip8_ig65m_from_scratch-9bae36ae.pth" to C:\Users\macca/.cache\torch\hub\checkpoints\r2plus1d_34_clip8_ig65m_from_scratch-9bae36ae.pth


Parametri totali:       63,754,631
Parametri addestrabili: 39,340,338 (61.71%)
Parametri congelati:    24,414,293

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 6.344 secondi

Architettura: MC3_18 | Pretrained: False
Caricamento MC3-18 da zero (no pretraining)...
Parametri totali:       11,752,896
Parametri addestrabili: 11,752,896 (100.00%)
Parametri congelati:    0

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 2.768 secondi

Architettura: S3D | Pretrained: False
Caricamento S3D da zero (no pretraining)...
Parametri totali:       8,434,848
Parametri addestrabili: 8,434,848 (100.00%)
Parametri congelati:    0

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 0.957 secondi

Architettura: X3D_S | Pretrained: False
Caricamento X3D_S...


Using cache found in C:\Users\macca/.cache\torch\hub\facebookresearch_pytorchvideo_main


Parametri totali:       4,023,762
Parametri addestrabili: 4,023,762 (100.00%)
Parametri congelati:    0

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 1.179 secondi

Architettura: MVIT_2D_IMAGENET | Pretrained: True
Caricamento MViTv2 2D preaddestrato su ImageNet-1K (Puro, No Kinetics) via TIMM...


Parametri totali:       34,494,944
Parametri addestrabili: 393,728 (1.14%)
Parametri congelati:    34,101,216

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 19.288 secondi

Architettura: SWIN_2D_IMAGENET | Pretrained: True
Caricamento Swin 2D preaddestrato su ImageNet-1K (Puro, No Kinetics) via TIMM...
Parametri totali:       49,230,986
Parametri addestrabili: 393,728 (0.80%)
Parametri congelati:    48,837,258

 ✅ Test Forward Pass completato con successo!
    📊 Shape dell'Output: torch.Size([8, 512]))
    ⏱️ Tempo di inferenza (CPU): 16.461 secondi


Tutte e 7 le architetture, per quanto internamente diversissime (Reti 3D custom, Reti fattorizzate, Reti Miste, e Transformer 2D), restituiscono esattamente la stessa shape: torch.Size([8, 512]) dimostrando grazie alla classe VideoEncoder per essere processato dal Discriminatore di Dominio (DANN).

- Con i modelli From Scratch (Handcrafted, MC3_18, S3D, X3D_S) 100% addestrabili e tutto il modello deve imparare dai piccoli domini UCF/HMDB.
- R(2+1)D_34_IG65M (61% addestrabili): sistema congela i primi blocchi (i più generici che riconoscono contorni e movimento) e sblocca l'ultimo blocco (layer4 e proiezione), evita di dimenticale le feature pre-addestrate su IG-65M ma permette alla rete di specializzarsi sulle 12 classi del problema.
- Transformer MViT e Swin (~1% addestrabili): poichè i Transformer hanno molti parametri che andrebbero in overfitting in 3 epoche se addestrati interamente sui dataset source, quindi è stato congelato il 99% della rete, lasciando addestrabile solo la "testa" lineare finale (circa 390.000 parametri). Estrae le feature visive perfette di ImageNet e fa Domain Adaptation solo sull'ultimissimo step.

Per concludere **S3D, Handcrafted e X3D_S** sono i modelli più leggeri (S3D soprattutto), mentre un buon compromesso è la R(2+1)D (più pesante, parametri e convoluzioni 3D piene ma con un tempo è accettabile per il boost di performance che daranno i pesi IG-65M), infine i due Transformer 2D (MViT e Swin) molto più lenti (poichè il codice prende il video da 16 frame, lo "smonta" in 16 foto distinte, fa passare ogni singola foto attraverso il gigantesco Transformer, e alla fine fa la media).


## Conclusioni

Come dimostrato dall'output soprastante:
1. Il nostro sistema adatta automaticamente le reti per generare uniformemente un **embedding da 512 dimensioni**, indipendentemente dall'architettura sottostante (ResNet, Inception, ecc.).
2. Il sistema di **congelamento parziale** lascia addestrabili solo una minima frazione di parametri proteggendo dal crollo delle prestazioni (overfitting) quando facciamo fine-tuning sui piccoli domini sorgente e permette alla loss avversariale di fare Domain Adaptation aggiornando solo le feature semantiche profonde.
3. Per rispettare il "no-leak" abbiamo scelto configurazioni **addestrati su altro** o **addestrati da zero** (pretrained=False). 

## Configurazioni Encoder

**Config A: Handcrafted + DANN**
- "Encoder addestrato from scratch. Nessun dato target usato in training supervisonato. Confronto onesto tra source-only e DA."

**Config B: R(2+1)D_34 moabitcoin/ig65m-pytorch + DANN**
- "I pesi dell'encoder sono stati pre-addestrati su IG-65M (video Instagram)."

**Config C: X3D_S from scratch + DANN**
- "Encoder più leggero, addestrato from scratch su HMDB+UCF. Nessun data leak."

**Config D: MVit 2D su ImageNet + DANN**
- "Encoder Trasformer 2D preaddestrati su immagini ImageNet